# Environment Setup

Configure a reproducible Python environment, load secrets from `.env`, and verify OpenAI API connectivity from Jupyter (Cursor / VS Code).


## 1. Overview

This notebook covers:

- Creating a Python 3.12+ virtual environment and installing dependencies from `requirements.txt`
- Storing `OPENAI_API_KEY` and `OPENAI_MODEL` in a gitignored `.env` file (never hardcode secrets)
- Selecting the project interpreter as the Jupyter kernel
- Running local health checks (runtime, packages, `.env`) without calling the API
- Sending a minimal **Responses API** request to confirm end-to-end connectivity

**How to use it:** complete §5 (terminal + health/key cells) first. Only run §6 once the key status is `SET`.


## 2. Motivation

Downstream notebooks and scripts depend on a consistent interpreter, installed packages, and a valid API credential. Failures that look like "the API is broken" are often an inactive venv, the wrong Jupyter kernel, a missing `.env`, or a placeholder key.

The standard pattern:

1. Isolate dependencies in a **virtual environment**
2. Keep secrets in **`.env`**, loaded via `python-dotenv`
3. Run notebooks against that same interpreter as the **Jupyter kernel**

Spending API tokens before those three are correct wastes time and hides the real problem. Local checks below catch setup issues first.


## 3. Concepts

### 3.1 Glossary

| Term | Meaning |
|------|--------|
| **Virtual environment (venv)** | Isolated Python install and packages for this project |
| **`requirements.txt`** | Dependency list installed with `pip install -r requirements.txt` |
| **API key** | Credential for OpenAI; treat like a password |
| **`.env` file** | Local `KEY=value` file loaded into process environment variables |
| **`python-dotenv`** | Loads `.env` via `load_dotenv()` for `os.getenv(...)` |
| **Jupyter kernel** | Interpreter that executes notebook cells; must match project `.venv` |
| **OpenAI client** | `OpenAI()` reads `OPENAI_API_KEY` from the environment |
| **Responses API** | `client.responses.create(...)` — current text/generation endpoint |

### 3.2 Runtime flow

1. Create `.venv` with `python -m venv .venv` and activate it.
2. Install packages from `requirements.txt`.
3. Copy `.env.example` → `.env` and set a real `OPENAI_API_KEY`.
4. Call `load_dotenv()` so variables from `.env` are available to the process.
5. Instantiate `OpenAI()`; it authenticates using `OPENAI_API_KEY`.
6. Call `client.responses.create(...)` for a connectivity check.
7. Ensure Cursor / VS Code uses the same `.venv` interpreter as the notebook kernel.

### 3.3 When to use this pattern

**Use for:** local development, demos, and services that need reproducible deps plus secrets.

**Avoid:** committing `.env`; pasting keys into chat or screenshots; sharing one long-lived key across teams without rotation; leaving secrets in notebook outputs.

**Production alternatives:** secret managers (AWS Secrets Manager, Azure Key Vault, GitHub Actions secrets) or host-injected environment variables — still never hardcode keys in source.


## 4. Architecture

### Setup flow

```mermaid
flowchart TD
    py[Python 3.12+] --> venv[Create and activate .venv]
    venv --> pip[pip install -r requirements.txt]
    pip --> env[Copy .env.example to .env]
    env --> key[Set OPENAI_API_KEY]
    key --> kernel[Select .venv as Jupyter kernel]
    kernel --> load[load_dotenv + OpenAI client]
    load --> ok[Health checks and first API call]
```

### Secret layout

```text
Project root
├── .env.example     ← template (safe to commit)
├── .env             ← real key (gitignored — never commit)
├── requirements.txt
└── notebooks/
    └── Environment_Setup.ipynb
            │
            ├─ load_dotenv()  → reads .env into os.environ
            └─ OpenAI()       → uses OPENAI_API_KEY
```
<div align="center">

<b>Python</b>
<br>↓<br>
<b>Virtual Environment</b>
<br>↓<br>
<b>Dependencies</b>
<br>↓<br>
<b>.env</b>
<br>↓<br>
<b>OPENAI_API_KEY</b>
<br>↓<br>
<b>Jupyter Kernel</b>
<br>↓<br>
<b>Environment Health Check</b>
<br>↓<br>
<b>OpenAI SDK</b>
<br>↓<br>
<b>Responses API</b>
<br>↓<br>
<b>First LLM Call</b>

</div>

## 5. Installation

The validation cells below need **no API key**. They check the interpreter, package imports, and `.env` presence without printing secrets.

### 5.1 One-time terminal setup

From the **project root**:

```bash
python -m venv .venv

# Windows (PowerShell)
.\.venv\Scripts\Activate.ps1

# macOS / Linux
source .venv/bin/activate

pip install -r requirements.txt

# Optional: register a named Jupyter kernel
python -m ipykernel install --user --name=project-venv --display-name="Project (.venv)"

# Create .env from the template
# Windows:  Copy-Item .env.example .env
# macOS/Linux:  cp .env.example .env
```

If `pip install` fails on Windows with `No such file or directory` / long-path errors, enable Win32 long paths, or install packages from `requirements.txt` in smaller batches. Cursor / VS Code only needs `.venv` + `ipykernel` to execute notebooks.

Edit `.env` and set a real key from [https://platform.openai.com/api-keys](https://platform.openai.com/api-keys). Prefer keeping the model name in `.env` too (see `.env.example`):

```text
OPENAI_API_KEY=sk-...
OPENAI_MODEL=gpt-4o-mini
OPENAI_TEMPERATURE=0
```

In Cursor / VS Code: open this notebook → **Select Kernel** → choose `.venv` (or **Project (.venv)**).

### 5.2 What the next cells do

1. **Health check** — Is this kernel the right Python? Are packages importable? Does `.env` exist?
2. **Key status** — Is `OPENAI_API_KEY` missing, still a placeholder, or set? (never prints the key)


In [25]:
# Local environment health check -- no API call required
# Confirms: project root, Python version, .env files, and package importability.
# Paths are printed relative to the project root (no full local directory URLs).
import importlib.util  # locate packages without importing them
import sys             # interpreter path + version
from pathlib import Path

# Notebooks often start in notebooks/; walk up until requirements.txt marks the repo root.
root = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]  # cwd, then parents
    if (p / "requirements.txt").is_file()  # project-root marker for this repo
)

def rel(path: Path | str) -> str:
    """Show path relative to project root; mask anything outside it."""
    p = Path(path).resolve()
    try:
        return str(p.relative_to(root)).replace("\\", "/") or "."
    except ValueError:
        return f"<outside-project>/{p.name}"

# Import names (dotenv == python-dotenv package). These must resolve in *this* kernel.
packages = ["openai", "dotenv", "tiktoken", "pandas", "numpy"]

print("=== Environment health check ===")
print(f"Python executable : {rel(sys.executable)}")  # e.g. .venv/Scripts/python.exe
print(f"Python version    : {sys.version.split()[0]}")  # major.minor.patch only
print(f"Project root      : {root.name}/  (path masked)")  # folder name only
print(f".env exists       : {(root / '.env').is_file()}")  # real secrets file (gitignored)
print(f".env.example      : {(root / '.env.example').is_file()}")  # committed template
print()

#  target is Python 3.12+
if sys.version_info < (3, 12):
    print(f"WARNING: Python 3.12+ recommended. Current: {sys.version_info.major}.{sys.version_info.minor}.")
else:
    print("OK: Python version meets 3.12+ requirement.")

print("\nPackage imports:")
missing = []  # collect names that fail so we can print one install hint
for name in packages:
    # find_spec avoids importing (and side effects); None means not installed for this interpreter
    ok = importlib.util.find_spec(name) is not None
    print(f"  {name:<12} {'OK' if ok else 'MISSING'}")
    if not ok:
        missing.append(name)

if missing:
    # Usually means wrong kernel or pip was run against system Python
    print("\nInstall dependencies from the project root:")
    print("  pip install -r requirements.txt")
else:
    print("\nOK: Required packages are importable in this kernel.")

if not (root / ".env").is_file():
    # Template is safe to commit; .env with the real key is not
    print("\nNext: copy .env.example to .env and set OPENAI_API_KEY.")


=== Environment health check ===
Python executable : <outside-project>/python.exe
Python version    : 3.13.5
Project root      : Setup_and_Prompt_Engineering/  (path masked)
.env exists       : True
.env.example      : True

OK: Python version meets 3.12+ requirement.

Package imports:
  openai       OK
  dotenv       OK
  tiktoken     OK
  pandas       OK
  numpy        OK

OK: Required packages are importable in this kernel.


In [26]:
# Inspect OPENAI_API_KEY presence without printing the secret
# Safe to run / share: only status labels are printed, never the key value or full local paths.
import os
from pathlib import Path

from dotenv import load_dotenv  # reads KEY=value lines into os.environ

root = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "requirements.txt").is_file()
)

# Explicit path beats bare load_dotenv() when cwd is notebooks/
load_dotenv(root / ".env")  # no-op if file missing; does not raise

key = os.getenv("OPENAI_API_KEY", "").strip()  # "" if unset; strip whitespace/quotes issues

# Classify without echoing the credential into notebook output
if not key:
    status = "MISSING -- set OPENAI_API_KEY in .env"
elif "your_openai_api_key" in key.lower():  # still the .env.example placeholder text
    status = "PLACEHOLDER -- replace the example value with a real key"
elif not key.startswith("sk-"):  # soft check: most OpenAI user keys use this prefix
    status = "UNEXPECTED FORMAT -- OpenAI keys usually start with sk-"
else:
    status = f"SET -- length {len(key)} chars (value hidden)"  # length only; never print key

print("Loaded .env from : .env  (project root; full path masked)")
print(f"OPENAI_API_KEY   : {status}")  # status label only


Loaded .env from : .env  (project root; full path masked)
OPENAI_API_KEY   : SET -- length 164 chars (value hidden)


## 6. OpenAI Connectivity

With a real key in `.env` and the project kernel selected, the next cell sends a minimal **Responses API** request.

This is the modern OpenAI text endpoint (`client.responses.create`). Compared with Chat Completions:

| Piece | Responses API |
|-------|----------------|
| System-style policy | `instructions=...` |
| User prompt | `input=...` |
| Reply text | `response.output_text` |
| Token usage | `usage.input_tokens` / `usage.output_tokens` |

Model and temperature come from `.env` (`OPENAI_MODEL`, `OPENAI_TEMPERATURE`).

Standard client bootstrap:

```python
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()          # or load_dotenv(project_root / ".env")
client = OpenAI()      # reads OPENAI_API_KEY from the environment
```


In [27]:
# Live OpenAI connectivity check -- requires a valid OPENAI_API_KEY in .env
# Uses the Responses API (not chat.completions). Costs a few tokens.
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import APIConnectionError, AuthenticationError, OpenAI, RateLimitError

root = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "requirements.txt").is_file()
)
load_dotenv(root / ".env")  # must run before OpenAI() so the key is visible

# Fail fast if the key is missing or still the .env.example placeholder
api_key = os.getenv("OPENAI_API_KEY", "")
if not api_key.strip() or "your_openai_api_key" in api_key.lower():
    raise SystemExit("Set OPENAI_API_KEY in .env before running this cell.")

# Prefer config from .env so the notebook stays portable across machines/models
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")  # fallback if OPENAI_MODEL unset
TEMPERATURE = float(os.getenv("OPENAI_TEMPERATURE", "0"))  # 0 = more deterministic
client = OpenAI()  # picks up OPENAI_API_KEY from the environment

try:
    # instructions ≈ system message; input ≈ user message; output_text is the reply string
    response = client.responses.create(
        model=MODEL,
        instructions="You are a concise assistant. Reply in one short sentence.",  # policy / persona
        input="Confirm that the OpenAI environment is working by saying: Environment setup OK.",  # task
        temperature=TEMPERATURE,
        max_output_tokens=40,  # keep the probe cheap
    )
    print("API call succeeded.")
    print(f"Model : {response.model}  (from OPENAI_MODEL={MODEL})")  # API may return a dated snapshot id
    print(f"Reply : {response.output_text}")  # convenience accessor for assistant text
    if response.usage:
        # Useful for budgeting; Responses API uses input_/output_ token names (not prompt_/completion_)
        print(
            f"Tokens: input={response.usage.input_tokens}, "
            f"output={response.usage.output_tokens}, "
            f"total={response.usage.total_tokens}"
        )
except AuthenticationError:
    # Bad, revoked, or malformed key
    print("Authentication failed. Check that OPENAI_API_KEY is valid and not revoked.")
except RateLimitError:
    # Too many requests or billing/quota issue
    print("Rate limit or quota hit. Wait and retry, or check billing/limits.")
except APIConnectionError as exc:
    # DNS, TLS, proxy, or offline
    print(f"Network error reaching OpenAI: {exc}")
except Exception as exc:
    # Anything else (model name typo, SDK mismatch, etc.)
    print(f"Request failed: {type(exc).__name__}: {exc}")


API call succeeded.
Model : gpt-4o-mini-2024-07-18  (from OPENAI_MODEL=gpt-4o-mini)
Reply : Environment setup OK.
Tokens: input=38, output=5, total=43


## 7. Implementation notes

1. **Project root** — Walk upward for `requirements.txt` so `.env` loads correctly when cwd is `notebooks/`.
2. **Health check** — Reports interpreter, Python version, `.env` presence, and package importability without spending API credits. Use `sys.executable` to confirm the kernel is `.venv`.
3. **Key status** — Reports MISSING / PLACEHOLDER / SET **without printing the key**. Never log `os.getenv("OPENAI_API_KEY")` in shared notebooks.
4. **Connectivity check** — Minimal `client.responses.create(...)`; typed errors for auth, rate limit, and network.
5. **Config from `.env`** — `OPENAI_MODEL` and `OPENAI_TEMPERATURE` drive the live call; do not hardcode the model name.
6. **Responses fields** — `instructions` + `input` in; `output_text` and `usage.input_tokens` / `usage.output_tokens` out.
7. **Path privacy** — Print project-relative paths (or folder name only); never commit notebook outputs with full local directory URLs.


## 8. Best practices

- Keep one venv per project; activate it before installing or running notebooks.
- Store secrets only in `.env` (or a secret manager); never in source, notebooks, or screenshots.
- Commit `.env.example`, not `.env`. Keep `.env` in `.gitignore`.
- Keep `OPENAI_MODEL` (and optional `OPENAI_TEMPERATURE`) in `.env` so notebooks stay config-driven.
- Select the project `.venv` as the Jupyter kernel before running cells.
- Prefer an explicit `.env` path from the project root when notebooks live in a subfolder.
- Validate locally first; only then spend tokens on a live API call.
- Rotate and revoke keys immediately if they leak.


## 9. Common failure modes

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| `ModuleNotFoundError` in notebook after `pip install` | Kernel ≠ venv where packages were installed | Select `.venv` as the kernel / interpreter |
| `AuthenticationError` | Missing, placeholder, or revoked key | Set a real `OPENAI_API_KEY` in project-root `.env` |
| Key "not found" though `.env` exists | `.env` under `notebooks/` or cwd mismatch | Place `.env` next to `.env.example`; load with absolute path |
| Packages install but imports still fail | `pip` ran against system Python | Activate `.venv` first, confirm `sys.executable` |
| Stale imports after recreating `.venv` | Old kernel still selected | Reselect interpreter; reinstall `ipykernel` if needed |

Also avoid printing `os.getenv("OPENAI_API_KEY")` in shared notebooks or committing outputs that contain secrets.


## 10. Validation checklist

1. Run the health-check cell and confirm Python ≥ 3.12, packages OK, and `.env` present.
2. Run the key-status cell and confirm status is `SET` (not MISSING / PLACEHOLDER).
3. Run the connectivity cell and confirm a successful model reply plus token usage.
4. Optionally run the aggregated readiness check below for a single pass/fail summary.


In [28]:
# Aggregated readiness check
# One-shot summary of version + critical packages + key. Does not call the API.
import importlib.util
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

root = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "requirements.txt").is_file()
)

problems = []  # empty list => ready; otherwise human-readable blockers

# Same bar as the health-check cell
if sys.version_info < (3, 12):
    problems.append(f"Python 3.12+ required; found {sys.version_info.major}.{sys.version_info.minor}")

# Minimum imports needed to talk to OpenAI from this notebook
for pkg in ("openai", "dotenv"):
    if importlib.util.find_spec(pkg) is None:
        problems.append(f"Package not importable: {pkg} (pip install -r requirements.txt)")

env_file = root / ".env"
if not env_file.is_file():
    problems.append(".env missing -- copy .env.example to .env")
else:
    load_dotenv(env_file)  # load only after we know the file exists
    key = os.getenv("OPENAI_API_KEY", "")
    if not key.strip():
        problems.append("OPENAI_API_KEY is empty")
    elif "your_openai_api_key" in key.lower():  # copied template but not edited
        problems.append("OPENAI_API_KEY is still the placeholder value")

if not problems:
    print("Environment ready.")  # safe to run the Responses API cell
else:
    print("Fix these issues:")
    for item in problems:
        print(f"  - {item}")


Environment ready.


## 11. Team onboarding (optional)

For a new engineer joining this repo, document:

- Exact commands for venv create/activate, `pip install -r requirements.txt`, and `.env` creation
- How to select the Jupyter kernel in Cursor / VS Code
- A verify step: run this notebook's health check + connectivity check
- A short troubleshooting table (wrong kernel, missing `.env`, placeholder key, auth error)

Do not include real API keys in that document.


## 12. Summary

- Use a **Python 3.12+ venv**, install from `requirements.txt`, and point Jupyter at that interpreter.
- Keep `OPENAI_API_KEY` and `OPENAI_MODEL` in a gitignored **`.env`** file; load with `python-dotenv` before creating `OpenAI()`.
- Validate locally (version, packages, key status), then run a minimal `client.responses.create(...)` call.
- Most connectivity failures are kernel, path, or placeholder-key issues — not model problems.

**Next:** `OpenAI_SDK.ipynb` for OpenAI Python SDK usage patterns.
